In [ ]:
# notebooks/03_exploratory_analysis.ipynb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

class RNASeqEDA:
    """Exploratory Data Analysis for RNA-Seq"""
    
    def __init__(self, expression_df, metadata_df):
        self.expression_df = expression_df
        self.metadata_df = metadata_df
        
    def plot_sample_distribution(self):
        """Plot distribution of samples by disease status"""
        plt.figure(figsize=(8, 6))
        disease_counts = self.metadata_df['disease_status'].value_counts()
        
        plt.bar(disease_counts.index, disease_counts.values, 
                color=['#3498db', '#e74c3c'])
        plt.xlabel('Disease Status', fontsize=12)
        plt.ylabel('Number of Samples', fontsize=12)
        plt.title('Sample Distribution', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('./results/figures/sample_distribution.png', dpi=300)
        plt.show()
        
    def perform_pca(self, n_components=2):
        """Perform PCA and visualize"""
        # Transpose: samples as rows, genes as columns
        X = self.expression_df.T
        
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X)
        
        # Create DataFrame for plotting
        pca_df = pd.DataFrame(
            X_pca, 
            columns=[f'PC{i+1}' for i in range(n_components)],
            index=X.index
        )
        
        # Merge with metadata
        pca_df = pca_df.merge(
            self.metadata_df[['sample_id', 'disease_status']], 
            left_index=True, 
            right_on='sample_id',
            how='left'
        )
        
        # Plot
        plt.figure(figsize=(10, 8))
        colors = {'COPD': '#e74c3c', 'Normal': '#3498db'}
        
        for disease, group in pca_df.groupby('disease_status'):
            plt.scatter(
                group['PC1'], 
                group['PC2'], 
                label=disease,
                c=colors.get(disease, '#95a5a6'),
                s=100,
                alpha=0.7,
                edgecolors='black',
                linewidth=0.5
            )
        
        plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', 
                   fontsize=12)
        plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', 
                   fontsize=12)
        plt.title('PCA: COPD vs Normal Samples', fontsize=14, fontweight='bold')
        plt.legend(fontsize=11)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig('./results/figures/pca_plot.png', dpi=300)
        plt.show()
        
        return pca, X_pca, pca_df
    
    def plot_variance_explained(self, n_components=20):
        """Plot cumulative variance explained by PCA components"""
        X = self.expression_df.T
        pca = PCA(n_components=n_components)
        pca.fit(X)
        
        cumulative_var = np.cumsum(pca.explained_variance_ratio_)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Individual variance
        ax1.bar(range(1, n_components+1), 
                pca.explained_variance_ratio_,
                color='#3498db',
                alpha=0.7)
        ax1.set_xlabel('Principal Component', fontsize=11)
        ax1.set_ylabel('Variance Explained', fontsize=11)
        ax1.set_title('Variance per Component', fontsize=12, fontweight='bold')
        
        # Cumulative variance
        ax2.plot(range(1, n_components+1), 
                 cumulative_var, 
                 marker='o',
                 color='#e74c3c',
                 linewidth=2)
        ax2.axhline(y=0.8, linestyle='--', color='gray', alpha=0.7)
        ax2.set_xlabel('Number of Components', fontsize=11)
        ax2.set_ylabel('Cumulative Variance Explained', fontsize=11)
        ax2.set_title('Cumulative Variance', fontsize=12, fontweight='bold')
        ax2.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('./results/figures/variance_explained.png', dpi=300)
        plt.show()
    
    def plot_hierarchical_clustering(self):
        """Hierarchical clustering of samples"""
        # Calculate correlation distance
        X = self.expression_df.T
        
        # Use correlation distance
        distances = pdist(X, metric='correlation')
        linkage_matrix = linkage(distances, method='average')
        
        plt.figure(figsize=(12, 8))
        
        # Create color map for disease status
        disease_colors = []
        color_map = {'COPD': '#e74c3c', 'Normal': '#3498db'}
        
        for sample in X.index:
            disease = self.metadata_df[
                self.metadata_df['sample_id'] == sample
            ]['disease_status'].values
            if len(disease) > 0:
                disease_colors.append(color_map.get(disease[0], '#95a5a6'))
            else:
                disease_colors.append('#95a5a6')
        
        # Plot dendrogram
        dendrogram(linkage_matrix, labels=X.index, 
                   leaf_rotation=90, leaf_font_size=8)
        plt.title('Hierarchical Clustering of Samples', 
                  fontsize=14, fontweight='bold')
        plt.xlabel('Sample ID', fontsize=12)
        plt.ylabel('Distance (Correlation)', fontsize=12)
        plt.tight_layout()
        plt.savefig('./results/figures/hierarchical_clustering.png', dpi=300)
        plt.show()
    
    def plot_top_variable_genes(self, top_n=50):
        """Plot heatmap of top variable genes"""
        # Calculate variance for each gene
        gene_var = self.expression_df.var(axis=1)
        top_genes = gene_var.nlargest(top_n).index
        
        # Subset data
        top_genes_df = self.expression_df.loc[top_genes]
        
        # Create color map for samples
        sample_colors = []
        for sample in top_genes_df.columns:
            disease = self.metadata_df[
                self.metadata_df['sample_id'] == sample
            ]['disease_status'].values
            if len(disease) > 0 and disease[0] == 'COPD':
                sample_colors.append('#e74c3c')
            else:
                sample_colors.append('#3498db')
        
        # Plot heatmap
        plt.figure(figsize=(14, 10))
        sns.clustermap(
            top_genes_df,
            cmap='RdBu_r',
            center=0,
            col_colors=sample_colors,
            figsize=(14, 10),
            cbar_kws={'label': 'Log2 Expression'},
            yticklabels=False
        )
        plt.suptitle(f'Top {top_n} Most Variable Genes', 
                     fontsize=14, fontweight='bold', y=1.0)
        plt.savefig('./results/figures/top_variable_genes_heatmap.png', 
                    dpi=300, bbox_inches='tight')
        plt.show()

# Usage
eda = RNASeqEDA(final_data, final_metadata)
eda.plot_sample_distribution()
pca_model, X_pca, pca_df = eda.perform_pca(n_components=2)
eda.plot_variance_explained(n_components=20)
eda.plot_top_variable_genes(top_n=50)
